# DuckPD Feature Store: Interactive Walkthrough

This notebook demonstrates how to connect to a partitioned Parquet feature store, align multi-rate time series without lookahead, access reference tables, stream training batches, and search catalog-declared embeddings without duplicating model configuration in application code.

### Core Highlights
- **Partition-Mirrored Caching**: Download only requested partitions and columns.
- **Metadata-Only Setup**: Construction may fetch catalog metadata, but feature partitions and model artifacts remain untouched until execution or explicit preparation.
- **Point-in-Time Correctness**: ASOF alignment applies declared availability delays.
- **Catalog-Inferred Semantic Search**: Verified vector metadata supplies the immutable model identity to `search_text()`.
- **Controlled Preparation**: Automatic preparation is bounded by store policy; explicit prewarming supports offline execution.
- **Lifecycle Observability**: Explain and profile expose origin, preparation, transfer, and inference behavior without exposing query text.


## Step 1: Import DuckPD and Load Credentials

First, we import DuckPD and read our `HF_TOKEN` from `.env`.

In [ ]:
import importlib
import os
import time
from datetime import timedelta
from pathlib import Path

import duckpd as pd

# Load token from the environment if the remote dataset requires one.
token = os.getenv("HF_TOKEN")
FEATURE_STORE_SOURCE = "hf://datasets/hifinab/fdb"
cache_dir = Path(".cache/fdb")

print(f"DuckPD version: {pd.__version__}")
print(f"HF_TOKEN configured: {bool(token)}")
print(f"Local cache path: {cache_dir.resolve()}")

## Step 2: Connect to the Remote Feature Store and Inspect the Catalog

`FeatureStore` reads and validates the compact catalog without transferring feature partitions or preparing embedding models. The explicit limits below also govern later catalog-driven model preparation.


In [ ]:
store = pd.FeatureStore(
    source=FEATURE_STORE_SOURCE,
    cache=cache_dir,
    token=token,
    auto_prepare_embeddings=True,
    embedding_prepare_timeout_seconds=300,
    embedding_download_limit_bytes=1_073_741_824,
)

catalog = store.catalog()
print(f"Catalog Name: {catalog.get('name')}")
print(f"Catalog Version: {catalog.get('catalog_version')}")
print(f"Datasets: {[d['name'] for d in catalog.get('datasets', [])]}")
timeseries = [dataset for dataset in catalog.get("datasets", []) if dataset["kind"] == "timeseries"]
partition_units = {dataset["name"]: dataset["partitioning"]["unit"] for dataset in timeseries}
assert set(partition_units.values()) == {"day"}, partition_units
print(f"Time-series partition units: {partition_units}")
print("Daily layout: <dataset>/year=YYYY/month=MM/day=DD/part.parquet")
print(f"Embedding models: {list(catalog.get('embedding_models', {}))}")
print()
print("Registered Features Sample:")
for feat_name, meta in list(catalog.get("features", {}).items())[:6]:
    delay = meta.get("availability_delay")
    safe = meta.get("lookahead_safe")
    print(f"  - {feat_name:20s} (Delay: {delay}, Safe: {safe})")

## Step 3: Access Static Dimension / Reference Tables

Feature stores often contain auxiliary reference data such as ticker symbology or listing exchanges. We access `store.table("symbology")`, which returns a first-class lazy `duckpd.DataFrame`.

In [ ]:
symbols = store.table("symbology")
symbols.sort_values("ticker").head(5)

## Step 4: Multi-Family Exact Alignment

Exact alignment performs an inner equi-join on matching event timestamps and series keys across datasets. Here, we retrieve minute close prices (`ohlcv:close`) alongside 200-bar moving averages (`sma:sma200`).

In [ ]:
t0 = time.perf_counter()
exact_features = store.features(
    features={
        "price": "ohlcv:close",
        "sma200": "sma:sma200",
    },
    start="2024-01-02T08:00:00Z",
    end="2024-01-02T09:00:00Z",
    filters={"ticker": ["001", "002"]},
    alignment="exact",
    order_by=["datetime", "ticker"],
)
exact_df = exact_features.collect()
print(f"Retrieved {len(exact_df)} rows in {time.perf_counter() - t0:.4f}s")
exact_df.head(6)

## Step 5: Point-in-Time (ASOF) Alignment with Availability Delays

In financial and event-driven machine learning, training models on data before it was physically available introduces **lookahead bias**.

In this catalog:
- `ohlcv:open` has availability delay `PT0S` (available immediately at bar open).
- `ohlcv:close` has availability delay `PT1M` (the close of the 08:00 bar is only known once the bar ends at 08:01).

With `alignment="point_in_time"` and `spine="ohlcv"`, DuckPD compiles a vectorized `ASOF LEFT JOIN`. The catalog's `history_lookback="P7D"` promises that seven days of predecessor partitions are sufficient at the query boundary. DuckPD therefore fetches only that bounded daily history instead of scanning every partition back to 2010.


In [ ]:
pit_features = store.features(
    features={
        "open": "ohlcv:open",
        "close": "ohlcv:close",
        "sma50": "sma:sma50",
        "sma200": "sma:sma200",
    },
    start="2024-01-02T08:00:00Z",
    end="2024-01-02T08:06:00Z",
    filters={"ticker": ["001"]},
    alignment="point_in_time",
    spine="ohlcv",
    order_by=["datetime"],
)
pit_df = pit_features.collect()
print(
    "Notice: At 08:00, 'close' is the latest eligible predecessor; "
    "the current 08:00 bar close becomes available at 08:01."
)
pit_df

## Step 6: Seamless DuckPD DataFrame Composition

Because `store.features()` returns a native `duckpd.DataFrame`, you can chain pandas-style operations lazily:
- Compute new indicators with `.assign()`
- Filter on signal conditions
- Merge with categorical symbology tables

In [ ]:
signals = pit_features.assign(
    trend=lambda df: df["close"] / df["sma200"],
    spread=lambda df: df["close"] - df["open"],
).merge(symbols, on="ticker", how="left")

signals_df = signals.collect()
signals_df[["datetime", "ticker", "company_name", "close", "trend", "spread"]]

## Step 7: Zero-Copy Arrow Batch Streaming for Machine Learning

For massive training sets that span multiple months or years, load chunked batches over consecutive time windows using `.feature_batches()` and stream directly to PyArrow record batches.

In [ ]:
batch_count = 0
row_count = 0
t0 = time.perf_counter()

for window_df in store.feature_batches(
    exact_features,
    window=timedelta(minutes=30),
    start="2024-01-02T08:00:00Z",
    end="2024-01-02T09:00:00Z",
):
    with window_df.to_arrow_batches(batch_size=10_000) as reader:
        for arrow_batch in reader:
            batch_count += 1
            row_count += arrow_batch.num_rows

elapsed = time.perf_counter() - t0
print(f"Streamed {batch_count} window batches ({row_count} rows) in {elapsed:.4f}s")

## Step 8: Inspect the Local Daily Cache Footprint

The cache mirrors the remote UTC-day hierarchy (`<dataset>/year=YYYY/month=MM/day=DD/part.parquet`) and stores only requested columns. Exact queries fetch intersecting days; point-in-time queries additionally fetch only the catalog-declared predecessor lookback.


In [ ]:
print(f"Cache Directory: {cache_dir.resolve()}")
for p in sorted(cache_dir.rglob("*")):
    if p.is_file():
        print(f"  {p.relative_to(cache_dir)}: {p.stat().st_size / 1e6:.2f} MB")

## Step 9: Inspect and Register the Catalog Model Runtime

The catalog owns the immutable embedding-space identity. `store.embedding_model()` is metadata-only. This catalog declares the Transformers backend, so the notebook registers an explicit CPU or available GPU runtime without preparing it yet. Install `transformers` and a matching PyTorch build first; the repository's ROCm environment described in `demo/generate_data/README.md` is supported.


In [ ]:
try:
    torch = importlib.import_module("torch")
    importlib.import_module("transformers")
except (ImportError, OSError) as error:
    raise RuntimeError(
        "Catalog semantic search requires transformers and a compatible PyTorch build. "
        "See demo/generate_data/README.md for the accelerator environment."
    ) from error

catalog_model = store.embedding_model("bge-small-en-v1.5")
embedding_device = "cuda" if torch.cuda.is_available() else "cpu"
store.session.register_embedding_provider(
    catalog_model,
    pd.TransformersEmbeddingProvider(catalog_model, device=embedding_device),
)

print(f"Model: {catalog_model.model}@{catalog_model.revision}")
print(f"Fingerprint: {catalog_model.fingerprint}")
print(f"Dimension/backend: {catalog_model.dimension}/{catalog_model.backend}")
print(f"Registered runtime device: {embedding_device}")
print(f"Prepared models before planning: {store.session.inspect_prepared_embedding_models()}")

## Step 10: Plan Catalog-Inferred Semantic Search

Request the news vector together with identifying fields, then omit `model=` from `search_text()`. DuckPD accepts that omission only because the `embedding` column carries verified catalog metadata. Planning remains lazy: it neither transfers the intersecting daily partition nor prepares the model nor embeds the query.


In [ ]:
news = store.features(
    features={
        "document_id": "news:document_id",
        "title": "news:title",
        "publisher": "news:publisher",
        "embedding": "news:embedding",
    },
    start="2024-01-02T08:00:00Z",
    end="2024-01-03T08:00:00Z",
    alignment="exact",
)

matches = news.vector.search_text(
    "AI chip demand and revenue growth",
    column="embedding",
    metric="cosine",
    k=5,
)[["document_id", "title", "publisher", "_distance"]]

logical_plan = matches.explain("json")
print(logical_plan)
assert '"model_origin": "catalog"' in logical_plan
assert '"model_prepared": false' in logical_plan
assert "AI chip demand and revenue growth" not in logical_plan

## Step 11: Execute, Profile, and Confirm Warm Reuse

The first profile is the execution boundary: automatic preparation happens before partition transfer. Reusing the same lazy plan reuses both the prepared model and cached query embedding. `ProfileResult.embedding_metrics` keeps catalog access, preparation, transfer, and query-inference lifecycle costs separate; `vector_operations` describes retrieval.


In [ ]:
cold_profile = matches.profile()
warm_profile = matches.profile()
result = matches.collect()

print("First execution embedding metrics:")
print(cold_profile.embedding_metrics)
print()
print("Warm execution embedding metrics:")
print(warm_profile.embedding_metrics)
print()
print("Vector operations:")
print(warm_profile.vector_operations)

display(result)

## Step 12: Explicitly Prewarm for Offline Execution

Production jobs can disable catalog-driven automatic preparation and prewarm during deployment instead. Preparation is eager; the later search remains lazy until `collect()`. Using a separate session prevents a stricter disabled policy from changing the earlier store's shared-session policy.


In [ ]:
with pd.connect() as offline_session:
    offline_store = pd.FeatureStore(
        source=FEATURE_STORE_SOURCE,
        cache=cache_dir,
        token=token,
        session=offline_session,
        auto_prepare_embeddings=False,
    )
    offline_model = offline_store.embedding_model("bge-small-en-v1.5")
    offline_session.register_embedding_provider(
        offline_model,
        pd.TransformersEmbeddingProvider(offline_model, device=embedding_device),
    )
    prepared = offline_session.prepare_embedding_model(offline_model)

    offline_news = offline_store.features(
        features={
            "document_id": "news:document_id",
            "title": "news:title",
            "embedding": "news:embedding",
        },
        start="2024-01-02T08:00:00Z",
        end="2024-01-03T08:00:00Z",
        alignment="exact",
    )
    offline_matches = offline_news.vector.search_text(
        "AI chip demand and revenue growth",
        column="embedding",
        k=3,
    )[["document_id", "title", "_distance"]]

    print(f"Prepared backend: {prepared.backend} via {prepared.execution_providers}")
    display(offline_matches.collect())

## Step 13: Close the Store Session

The store owns its session because no session was supplied at construction.


In [ ]:
store.session.close()